In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import muon as mu
from muon import atac as ac
from pyjaspar import jaspardb
import pychromvar as pc


In [2]:
loc_mm10 = '/data2st1/junyi/ref/GRCm38.p6.genome.fa'

In [9]:
jdb_obj = jaspardb(release='JASPAR2026')
motifs = jdb_obj.fetch_motifs(
    collection = 'CORE',
    tax_group = ['vertebrates'])


In [3]:
adata=sc.read_h5ad('/data1st2/junyi/output/atac1112/subset/region_nt/PFC_PFC_GABA.h5ad')

In [7]:
adata.var['chr'] = adata.var.index
adata.var.index = adata.var['chr'].str.replace(':', '-')
adata.X = adata.layers['count'] # use norm matrix for chromvar

In [8]:
pc.add_peak_seq(adata, genome_file=loc_mm10)


100%|██████████| 1660766/1660766 [00:10<00:00, 152519.55it/s]


In [10]:
pc.match_motif(adata, motifs=motifs)

100%|██████████| 1660766/1660766 [20:51<00:00, 1327.36it/s]


In [11]:
df_motif_match = pd.DataFrame(adata.varm['motif_match'], index=adata.var_names, columns=adata.uns['motif_name'])

In [12]:
df_motif_match.to_csv('/data2st1/junyi/output/atac1112/cCRE/motif_match.csv')

In [13]:
df_motif_match

,MA0004.1.Arnt,MA0069.1.PAX6,MA0071.1.RORA,MA0074.1.RXRA::VDR,MA0101.1.REL,MA0107.1.RELA,MA0111.1.Spz1,MA0115.1.NR1H2::RXRA,MA0119.1.NFIC::TLX1,MA0130.1.ZNF354C,...,MA0116.2.ZNF423,MA2484.1.Dmrtb1,MA2485.1.Duxbl1,MA2486.1.Fosb,MA1540.3.Nr5a1,MA0505.3.Nr5a2,MA0506.3.Nrf1,MA1618.2.Ptf1A,MA0002.3.Runx1,MA2503.1.Banp
chr,,,,,,,,,,,,,,,,,,,,,
chr1-3003518-3004019,0,0,0,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,0
chr1-3007481-3007982,0,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
chr1-3012480-3012981,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
chr1-3013424-3013925,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
chr1-3014717-3015218,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chrY-90812660-90813161,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
chrY-90811433-90811934,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
chrY-90813596-90814097,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [44]:
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)


In [49]:
mu.pp.filter_var(adata, 'n_cells_by_counts', lambda x: x >= 50)
mu.pp.filter_obs(adata, 'n_genes_by_counts', lambda x: (x >= 2000) )
mu.pp.filter_obs(adata, 'total_counts', lambda x: (x >= 4000))


In [50]:
adata

AnnData object with n_obs × n_vars = 6326 × 642438
    obs: 'sample', 'doublet_probability', 'doublet_score', 'leiden', 'leiden_default', 'leiden_res_0.1', 'leiden_res_0.2', 'leiden_res_0.3', 'leiden_res_0.4', 'leiden_res_0.5', 'leiden_res_0.6', 'leiden_res_0.7', 'leiden_res_0.8', 'leiden_res_0.9', 'leiden_res_1.0', 'leiden_res_1.1', 'leiden_res_1.2', 'leiden_res_1.3', 'leiden_res_1.4', 'leiden_res_1.5', 'leiden_res_1.6', 'leiden_res_1.7', 'leiden_res_1.8', 'leiden_res_1.9', 'celltype.L2.Condition', 'celltype.L1', 'celltype.L2', 'Neurotransmitter_celltype', 'celltype.L1_ct', 'Sample_name', 'Condition', 'Region', 'celltype.L2.raw', 'region_nt', 'celltype.L3', 'celltype.L4', 'celltype.L2.refined', 'expriment', 'n_genes_by_counts', 'total_counts'
    var: 'chr', 'gc_bias', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'log1p', 'peak_seq'
    obsm: 'X_spectral', 'X_umap'
    layers: 'count'

100%|██████████| 642438/642438 [00:02<00:00, 215873.24it/s]


In [53]:
pc.add_gc_bias(adata)

100%|██████████| 642438/642438 [00:03<00:00, 190363.66it/s]
/home/junyichen/anaconda3/envs/scenicplus/lib/python3.11/site-packages/pychromvar/preprocessing.py:134: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  adata.var['gc_bias'] = bias


In [54]:
pc.get_bg_peaks(adata)


/home/junyichen/anaconda3/envs/scenicplus/lib/python3.11/site-packages/scipy/sparse/_index.py:145: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)


In [59]:
adata.X = adata.X.astype(np.float32)

100%|██████████| 642438/642438 [07:28<00:00, 1433.61it/s]


In [71]:
len(adata.uns['motif_name'])

1019

In [75]:
adata.varm['motif_match'].shape

(642438, 1019)

In [78]:
dev = pc.compute_deviations(adata)


2026-06-08 15:15:29 INFO     computing expectation reads per cell and peak...
2026-06-08 15:20:10 INFO     computing observed motif deviations...
2026-06-08 15:20:38 INFO     computing background deviations...
/home/junyichen/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


In [85]:
dev.var

""
MA0004.1.Arnt
MA0069.1.PAX6
MA0071.1.RORA
MA0074.1.RXRA::VDR
MA0101.1.REL
...
MA0505.3.Nr5a2
MA0506.3.Nrf1
MA1618.2.Ptf1A
MA0002.3.Runx1
